# Error Functions in Machine Learning

An **error function** (also called a loss or cost function) tells us how far off our model's predictions are from the true answers. The goal of training is to **minimize** this error.

Think of it like a grade on a test — the error function scores how badly the model performed, and the model learns by trying to lower that score.

## Binary Classification Error

$$E = -\frac{1}{m} \sum_{i=1}^{m} (1 - y_i) \ln(1 - \hat{y}_i) + y_i \ln(\hat{y}_i)$$

This is the **cross-entropy loss** for binary classification (2 classes: 0 or 1).

| Symbol | Meaning |
|--------|---------|
| $m$ | Total number of training samples |
| $y_i$ | True label for sample $i$ (either 0 or 1) |
| $\hat{y}_i$ | Model's predicted probability that sample $i$ is class 1 |
| $\frac{1}{m}$ | Averages the error over all samples |

**How it works:** For each sample, only one term is "active":
- If $y_i = 1$: we compute $\ln(\hat{y}_i)$ → penalizes predictions far from 1
- If $y_i = 0$: we compute $\ln(1 - \hat{y}_i)$ → penalizes predictions far from 0

## Total Error as a Function of Weights and Bias

$$E(W, b) = -\frac{1}{m} \sum_{i=1}^{m} (1 - y_i) \ln(1 - \sigma(Wx^{(i)} + b)) + y_i \ln(\sigma(Wx^{(i)} + b))$$

This is the same error function, but written to show **what the model actually computes**:

| Symbol | Meaning |
|--------|---------|
| $W$ | Weights — how much each input feature matters |
| $b$ | Bias — a constant offset |
| $x^{(i)}$ | The input features for sample $i$ |
| $\sigma(\cdot)$ | The sigmoid function — squishes any number into the range (0, 1) |

The flow is: `inputs → linear combination (Wx + b) → sigmoid → probability → error`

This form is useful because **gradient descent** needs to know how the error depends on $W$ and $b$ to update them.

## Multiclass Error

$$E = -\frac{1}{m} \sum_{i=1}^{m} \sum_{j=1}^{n} y_{ij} \ln(\hat{y}_{ij})$$

When there are more than 2 classes (e.g., recognizing digits 0–9):

| Symbol | Meaning |
|--------|---------|
| $n$ | Number of classes |
| $y_{ij}$ | 1 if sample $i$ belongs to class $j$, else 0 (one-hot encoded) |
| $\hat{y}_{ij}$ | Model's predicted probability that sample $i$ is class $j$ |

The double sum goes over every sample AND every class. For each sample, only the correct class contributes (since $y_{ij} = 0$ for wrong classes).

## Derivative of the Sigmoid Function

$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

This is the **derivative** of the sigmoid function, which is essential for **backpropagation** — the algorithm used to update weights during training.

| Symbol | Meaning |
|--------|---------|
| $\sigma(x)$ | The sigmoid function: $\sigma(x) = \frac{1}{1 + e^{-x}}$ |
| $\sigma'(x)$ | The rate of change of the sigmoid at point $x$ |

**Why it matters:** During gradient descent, we need to compute how the error changes with respect to the weights. Since the sigmoid is part of the model, its derivative appears in the chain rule calculations.

**Key property:** The derivative is expressed entirely in terms of the sigmoid itself, making it computationally efficient — no need to recalculate from scratch.

**Note:** The maximum value of $\sigma'(x)$ is $0.25$, occurring at $x = 0$. This means gradients can become very small for large $|x|$, which is known as the **vanishing gradient problem**.


## Gradient of the Error Function

$$\nabla E = \left( \frac{\partial E}{\partial w_1}, \cdots, \frac{\partial E}{\partial w_n}, \frac{\partial E}{\partial b} \right)$$

The **gradient** is a vector of all partial derivatives of the error function with respect to each parameter. It points in the direction of **steepest increase** of the error.

| Symbol | Meaning |
|--------|---------|
| $\nabla E$ | The gradient vector of the error function |
| $\frac{\partial E}{\partial w_j}$ | How much the error changes when weight $w_j$ changes slightly |
| $\frac{\partial E}{\partial b}$ | How much the error changes when the bias $b$ changes slightly |
| $n$ | Number of input features (and corresponding weights) |

**Why it matters:** In **gradient descent**, we update the weights and bias by moving in the **opposite direction** of the gradient to minimize the error:

$$w_j \leftarrow w_j - \alpha \frac{\partial E}{\partial w_j}, \quad b \leftarrow b - \alpha \frac{\partial E}{\partial b}$$

where $\alpha$ is the **learning rate** — a hyperparameter that controls the step size.

**Key insight:** The gradient tells the model *which direction* to adjust each parameter and *by how much*, making it the core mechanism behind learning.

## Gradient Calculation — Step-by-Step Derivation

Now that we know the gradient is the vector of partial derivatives, let's actually **compute** it step by step.

To simplify the calculation, we focus on the error produced by a **single point** first. The total error is just the average of all individual errors, so once we know the derivative for one point, we can average over all $m$ points.

The error produced by a single point is:

$$E = -y \ln(\hat{y}) - (1 - y) \ln(1 - \hat{y})$$

where the prediction is $\hat{y} = \sigma(Wx + b)$.

Our goal is to compute the partial derivatives $\frac{\partial E}{\partial w_j}$ and $\frac{\partial E}{\partial b}$.

### Step 1: Derivative of $\hat{y}$ with respect to $w_j$

Since $\hat{y} = \sigma(Wx + b)$, we apply the **chain rule**. Recall that $\sigma'(x) = \sigma(x)(1 - \sigma(x))$, so:

$$\frac{\partial}{\partial w_j} \hat{y} = \frac{\partial}{\partial w_j} \sigma(Wx + b) = \sigma'(Wx + b) \cdot \frac{\partial}{\partial w_j}(Wx + b)$$

Now, expanding $Wx + b = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$, the only term that depends on $w_j$ is $w_j x_j$, so:

$$\frac{\partial}{\partial w_j}(Wx + b) = x_j$$

Putting it together:

$$\frac{\partial \hat{y}}{\partial w_j} = \sigma'(Wx + b) \cdot x_j = \hat{y}(1 - \hat{y}) \cdot x_j$$

### Step 2: Derivative of $E$ with respect to $w_j$

Now we compute $\frac{\partial E}{\partial w_j}$ using the chain rule on $E = -y \ln(\hat{y}) - (1 - y) \ln(1 - \hat{y})$:

$$\frac{\partial E}{\partial w_j} = -y \cdot \frac{1}{\hat{y}} \cdot \frac{\partial \hat{y}}{\partial w_j} - (1 - y) \cdot \frac{1}{1 - \hat{y}} \cdot (-1) \cdot \frac{\partial \hat{y}}{\partial w_j}$$

Substituting $\frac{\partial \hat{y}}{\partial w_j} = \hat{y}(1 - \hat{y}) \cdot x_j$ from Step 1:

$$= -y \cdot \frac{1}{\hat{y}} \cdot \hat{y}(1 - \hat{y}) \cdot x_j + (1 - y) \cdot \frac{1}{1 - \hat{y}} \cdot \hat{y}(1 - \hat{y}) \cdot x_j$$

$$= -y(1 - \hat{y}) \cdot x_j + (1 - y)\hat{y} \cdot x_j$$

$$= \left[-y + y\hat{y} + \hat{y} - y\hat{y}\right] x_j$$

$$= (\hat{y} - y) \cdot x_j = -(y - \hat{y}) \cdot x_j$$

**Result:**

$$\frac{\partial E}{\partial w_j} = -(y - \hat{y}) \cdot x_j$$

### Step 3: Derivative of $E$ with respect to $b$

A similar calculation (replacing $w_j$ with $b$) gives us:

$$\frac{\partial E}{\partial b} = -(y - \hat{y})$$

This follows the same logic as Step 2, except that $\frac{\partial}{\partial b}(Wx + b) = 1$ instead of $x_j$.

### Step 4: The Full Gradient — A Beautiful Result

Combining the results from Steps 2 and 3, for a point with coordinates $(x_1, \ldots, x_n)$, label $y$, and prediction $\hat{y}$, the gradient of the error function at that point is:

$$\nabla E = \left(-(y - \hat{y})x_1, \;\cdots, \;-(y - \hat{y})x_n, \;-(y - \hat{y})\right)$$

This can be written compactly as:

$$\nabla E = -(y - \hat{y})(x_1, \ldots, x_n, 1)$$

**Why this is fascinating:** The gradient is a **scalar** times the coordinates of the point (with a 1 appended for the bias). And what is that scalar? It's $-(y - \hat{y})$ — a multiple of the **difference between the label and the prediction**.

| Scenario | $(y - \hat{y})$ | Effect |
|----------|-----------------|--------|
| Label is 1, prediction close to 0 | Large positive | Large gradient → big update toward the point |
| Label is 0, prediction close to 1 | Large negative | Large gradient → big update away from the point |
| Prediction is close to label | Near zero | Small gradient → tiny update (already doing well) |

**Intuition:** A misclassified point produces a large gradient, pushing the model to correct its mistake. A correctly classified point barely affects the weights. This is exactly the behavior we want from a learning algorithm.

### Gradient Descent Step

So, a small gradient means we'll change our coordinates by a little bit, and a large gradient means we'll change our coordinates by a lot. If this sounds anything like the perceptron algorithm, this is no coincidence!

Since the gradient descent step consists in **subtracting a multiple of the gradient** of the error function at every point, the weights update as:

$$w_i' \leftarrow w_i - \alpha \left[-(y - \hat{y})x_i\right]$$

which simplifies to:

$$w_i' \leftarrow w_i + \alpha (y - \hat{y}) x_i$$

Similarly, the bias updates as:

$$b' \leftarrow b + \alpha (y - \hat{y})$$

| Symbol | Meaning |
|--------|---------|
| $w_i$ | Current weight for feature $i$ |
| $w_i'$ | Updated weight for feature $i$ after one gradient descent step |
| $\alpha$ | Learning rate — controls the step size of each update |
| $y$ | True label for the sample (0 or 1) |
| $\hat{y}$ | Model's predicted probability for the sample |
| $x_i$ | The $i$-th input feature of the sample |
| $b$ | Current bias |
| $b'$ | Updated bias after one gradient descent step |

**Note:** Since we've taken the average of the errors, the term we are adding should technically be $\frac{1}{m} \cdot \alpha$ instead of $\alpha$. But since $\alpha$ is a constant, we simplify by absorbing $\frac{1}{m}$ into the learning rate and just call the whole thing $\alpha$.

### Logistic Regression Algorithm

Here are our steps for logistic regression:

1. Start with random weights: $w_1, \ldots, w_n, b$
2. For every point $(x_1, \ldots, x_n)$:
    - For $i = 1 \ldots n$:
      - Update $w_i' \leftarrow w_i + \alpha (y - \hat{y}) x_i$
    - Update $b' \leftarrow b + \alpha (y - \hat{y})$
3. Repeat until the error is small


### Gradient Descent vs Perceptron Algorithm

**Gradient Descent**

With gradient descent, we change the weights from $w_i$ to $w_i + \alpha (y - \hat{y}) x_i$.

**Perceptron Algorithm**

With the perceptron algorithm we only change the weights on the misclassified points. If a point $x$ is misclassified:

We change $w_i$:

- To $w_i + \alpha x_i$ if positive
- To $w_i - \alpha x_i$ if negative

If correctly classified: $y - \hat{y} = 0$

If misclassified:

- $y - \hat{y} = 1$ if positive
- $y - \hat{y} = -1$ if negative

## Feedforward

Feedforward is the process neural networks use to turn an input into an output. It works in three steps:

1. Take the input vector
2. Apply a sequence of linear transformations and sigmoid activations
3. Produce the final prediction

### Feedforward Formula (one hidden layer)

$$\hat{y} = \sigma\left(W^{(2)} \cdot \sigma\left(W^{(1)} x + b^{(1)}\right) + b^{(2)}\right)$$

| Symbol | Meaning |
|--------|---------|
| $x = \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$ | Input vector (2 features) |
| $W^{(1)} = \begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \end{pmatrix}$ | Weights from input to hidden layer |
| $b^{(1)} = \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \end{pmatrix}$ | Biases for the hidden layer |
| $W^{(2)} = \begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} \end{pmatrix}$ | Weights from hidden layer to output |
| $b^{(2)}$ | Bias for the output layer |
| $\sigma$ | Sigmoid activation function |

**How it flows:** input $x$ → multiply by $W^{(1)}$, add $b^{(1)}$ → apply $\sigma$ → multiply by $W^{(2)}$, add $b^{(2)}$ → apply $\sigma$ → prediction $\hat{y}$

#### Expanded — with all vectors written out

$$\hat{y} = \sigma\left(\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} \end{pmatrix} \cdot \sigma\left(\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \end{pmatrix}\right) + b^{(2)}\right)$$

Reading inside-out:

1. $\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \end{pmatrix}$ — linear combination in the hidden layer (produces a 2-element vector)

2. $\sigma(\cdots)$ — apply sigmoid to each element of that vector

3. $\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} \end{pmatrix} \cdot (\text{hidden output}) + b^{(2)}$ — linear combination in the output layer (produces a scalar)

4. $\sigma(\cdots)$ — apply sigmoid to get the final prediction $\hat{y}$

### Multi-Layer Perceptron (multiple hidden layers)

$$\hat{y} = \sigma\Big(W^{(3)} \cdot \sigma\Big(W^{(2)} \cdot \sigma\Big(W^{(1)} x\Big)\Big)\Big)$$

This is the same idea extended to deeper networks — just stack more layers of (linear transformation → sigmoid) on top of each other. Each layer combines the previous layer's outputs in new ways, allowing the network to learn increasingly complex patterns.

#### Expanded — with biases included

$$\hat{y} = \sigma\Big(W^{(3)} \cdot \sigma\Big(W^{(2)} \cdot \sigma\Big(W^{(1)} x + b^{(1)}\Big) + b^{(2)}\Big) + b^{(3)}\Big)$$

Reading inside-out:

1. $W^{(1)} x + b^{(1)}$ — linear transformation at layer 1
2. $\sigma(\cdots)$ — activate layer 1 output
3. $W^{(2)} \cdot (\text{layer 1 output}) + b^{(2)}$ — linear transformation at layer 2
4. $\sigma(\cdots)$ — activate layer 2 output
5. $W^{(3)} \cdot (\text{layer 2 output}) + b^{(3)}$ — linear transformation at output layer
6. $\sigma(\cdots)$ — final activation → prediction $\hat{y}$

Each layer has its own weight matrix $W^{(k)}$ and bias vector $b^{(k)}$, so the total number of learnable parameters grows with each added layer.

#### Expanded — with all vectors written out

For a network with 2 inputs, 3 hidden neurons in layer 1, 2 hidden neurons in layer 2, and 1 output:

$$\hat{y} = \sigma\left(\begin{pmatrix} W_{11}^{(3)} & W_{21}^{(3)} \end{pmatrix} \cdot \sigma\left(\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} & W_{31}^{(2)} \\ W_{12}^{(2)} & W_{22}^{(2)} & W_{32}^{(2)} \end{pmatrix} \cdot \sigma\left(\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \\ W_{13}^{(1)} & W_{23}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \\ b_3^{(1)} \end{pmatrix}\right) + \begin{pmatrix} b_1^{(2)} \\ b_2^{(2)} \end{pmatrix}\right) + b^{(3)}\right)$$

Reading inside-out:

1. $\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \\ W_{13}^{(1)} & W_{23}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \\ b_3^{(1)} \end{pmatrix}$ — layer 1: 2 inputs → 3 outputs

2. $\sigma(\cdots)$ — activate each of the 3 outputs

3. $\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} & W_{31}^{(2)} \\ W_{12}^{(2)} & W_{22}^{(2)} & W_{32}^{(2)} \end{pmatrix} \cdot (\text{layer 1 output}) + \begin{pmatrix} b_1^{(2)} \\ b_2^{(2)} \end{pmatrix}$ — layer 2: 3 inputs → 2 outputs

4. $\sigma(\cdots)$ — activate each of the 2 outputs

5. $\begin{pmatrix} W_{11}^{(3)} & W_{21}^{(3)} \end{pmatrix} \cdot (\text{layer 2 output}) + b^{(3)}$ — output layer: 2 inputs → 1 scalar

6. $\sigma(\cdots)$ — final activation → prediction $\hat{y}$

**Notice the pattern:** the matrix dimensions match the connections — $W^{(k)}$ has shape (neurons in layer $k$) × (neurons in layer $k-1$).

## Backpropagation

Backpropagation is the algorithm that computes the gradient of the error function with respect to **every weight in the network** — across all layers.

The key idea: we already know how to compute the gradient for the output layer (same as logistic regression). The challenge is computing the gradient for the **hidden layers**, where weights don't directly produce the prediction. Backpropagation solves this by propagating the error **backwards** from the output, layer by layer.

**Why "back"?** In feedforward, data flows forward: input → hidden → output. In backpropagation, gradients flow backward: output → hidden → input.

### The Chain Rule for Multi-Layer Networks

For a single-layer model, we computed:

$$\frac{\partial E}{\partial w_j} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w_j}$$

For a multi-layer network, the chain rule extends. Consider a 2-layer network where:
- Layer 1 produces hidden outputs: $h = \sigma(W^{(1)} x + b^{(1)})$
- Layer 2 produces the prediction: $\hat{y} = \sigma(W^{(2)} h + b^{(2)})$

To find how the error depends on a weight in **layer 2** ($W^{(2)}$), the chain is short:

$$\frac{\partial E}{\partial W^{(2)}} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial W^{(2)}}$$

To find how the error depends on a weight in **layer 1** ($W^{(1)}$), the chain is longer — it must pass through layer 2:

$$\frac{\partial E}{\partial W^{(1)}} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial h} \cdot \frac{\partial h}{\partial W^{(1)}}$$

| Symbol | Meaning |
|--------|---------|
| $h$ | Output of the hidden layer (a vector) |
| $\frac{\partial E}{\partial \hat{y}}$ | How the error changes with the prediction (we already know this) |
| $\frac{\partial \hat{y}}{\partial h}$ | How the prediction changes with the hidden outputs |
| $\frac{\partial h}{\partial W^{(1)}}$ | How the hidden outputs change with layer 1 weights |

**Key insight:** Each link in the chain is a derivative we can compute. The chain rule lets us multiply them together to "reach back" to any layer.

### Step 1: Gradient at the Output Layer

We define the **error signal** at the output as $\delta^{(2)}$ — how much the output node is "responsible" for the error.

For the output layer, this is the same result we derived earlier for logistic regression:

$$\delta^{(2)} = \hat{y} - y$$

Then the gradients for the output layer weights and bias are:

$$\frac{\partial E}{\partial W_{j}^{(2)}} = \delta^{(2)} \cdot h_j$$

$$\frac{\partial E}{\partial b^{(2)}} = \delta^{(2)}$$

| Symbol | Meaning |
|--------|---------|
| $\delta^{(2)}$ | Error signal at the output layer |
| $\hat{y}$ | Model's prediction (output of the last sigmoid) |
| $y$ | True label |
| $h_j$ | The $j$-th output of the hidden layer (input to this layer) |

**This is familiar:** the gradient is the error signal times the input to that layer — same pattern as the single-layer case, just using hidden outputs $h_j$ instead of raw inputs $x_j$.

### Step 2: Gradient at the Hidden Layer — The "Back" Part

Now the new piece: to update $W^{(1)}$, we need to know how much each hidden neuron contributed to the error. We compute the **error signal** for each hidden neuron $j$:

$$\delta_j^{(1)} = \left(\sum_{k} W_{jk}^{(2)} \cdot \delta_k^{(2)}\right) \cdot h_j(1 - h_j)$$

For our 2-layer network with one output, this simplifies to:

$$\delta_j^{(1)} = W_{j}^{(2)} \cdot \delta^{(2)} \cdot h_j(1 - h_j)$$

| Part | What it does |
|------|-------------|
| $W_{j}^{(2)} \cdot \delta^{(2)}$ | Takes the output error and weights it by how much hidden neuron $j$ contributed (via $W_j^{(2)}$) |
| $h_j(1 - h_j)$ | The sigmoid derivative at hidden neuron $j$ — scales the error by how sensitive the sigmoid was at that point |

Then the gradients for the hidden layer weights and bias are:

$$\frac{\partial E}{\partial W_{ij}^{(1)}} = \delta_j^{(1)} \cdot x_i$$

$$\frac{\partial E}{\partial b_j^{(1)}} = \delta_j^{(1)}$$

**Same pattern again:** the gradient is the error signal times the input to that layer. The only difference is how $\delta$ is computed — at the output it's $(\hat{y} - y)$, at hidden layers it's propagated back from the layer above.

### Step 3: The General Pattern

For **any** layer $k$ in a network with $L$ layers:

$$\delta^{(L)} = \hat{y} - y \quad \text{(output layer)}$$

$$\delta^{(k)} = \left({W^{(k+1)}}^T \cdot \delta^{(k+1)}\right) \odot \sigma'(z^{(k)}) \quad \text{(hidden layers)}$$

where $z^{(k)} = W^{(k)} h^{(k-1)} + b^{(k)}$ is the value **before** the sigmoid, and $\odot$ means element-wise multiplication.

Then for any layer:

$$\frac{\partial E}{\partial W^{(k)}} = \delta^{(k)} \cdot {h^{(k-1)}}^T$$

$$\frac{\partial E}{\partial b^{(k)}} = \delta^{(k)}$$

| Symbol | Meaning |
|--------|---------|
| $\delta^{(k)}$ | Error signal at layer $k$ — how much each neuron in layer $k$ contributed to the error |
| ${W^{(k+1)}}^T$ | Transpose of the weight matrix from the next layer — used to "send back" the error |
| $\odot$ | Element-wise (Hadamard) product — multiply corresponding elements |
| $\sigma'(z^{(k)})$ | Sigmoid derivative at layer $k$, i.e., $h^{(k)} \odot (1 - h^{(k)})$ |
| $z^{(k)}$ | Pre-activation value: $W^{(k)} h^{(k-1)} + b^{(k)}$ (before applying $\sigma$) |
| $h^{(k-1)}$ | Output of the previous layer (input to layer $k$). For $k=1$, this is $x$ |

### The Backpropagation Algorithm

Putting feedforward and backpropagation together, here is the full training loop:

**1. Feedforward** — compute all outputs, layer by layer:

$$z^{(k)} = W^{(k)} h^{(k-1)} + b^{(k)}, \quad h^{(k)} = \sigma(z^{(k)})$$

**2. Output error** — compare prediction with true label:

$$\delta^{(L)} = \hat{y} - y$$

**3. Backpropagate** — compute error signals from last hidden layer back to first:

$$\delta^{(k)} = \left({W^{(k+1)}}^T \cdot \delta^{(k+1)}\right) \odot h^{(k)} \odot (1 - h^{(k)})$$

**4. Update** — adjust all weights and biases using the error signals:

$$W^{(k)} \leftarrow W^{(k)} - \alpha \cdot \delta^{(k)} \cdot {h^{(k-1)}}^T$$

$$b^{(k)} \leftarrow b^{(k)} - \alpha \cdot \delta^{(k)}$$

**5. Repeat** — go back to step 1 with the next training sample (or batch)

---

**The beautiful symmetry:**

| Direction | What flows | Algorithm |
|-----------|-----------|-----------|
| Forward (input → output) | Data: $x \to h^{(1)} \to h^{(2)} \to \cdots \to \hat{y}$ | Feedforward |
| Backward (output → input) | Errors: $\delta^{(L)} \to \delta^{(L-1)} \to \cdots \to \delta^{(1)}$ | Backpropagation |

The network learns by alternating these two passes: forward to make a prediction, backward to figure out how to improve it.